# 🎯 SAM3 Auto-Labeling
**SAM3PromptableConceptImageSegmenter** — text prompt ile nesne tespiti → YOLO `.txt` etiket

## Kaggle Inputs
| Slot | Tip | Açıklama |
|------|-----|----------|
| `keras/sam3` | Model | SAM3 Keras modeli |
| görüntü dataset | Dataset | Etiketlenecek görüntüler |

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────────────────
# keras_hub'ı YÜKSELTME — Kaggle'ın default versiyonu SAM3 destekler
import keras_hub
import keras
import numpy as np
from pathlib import Path
from PIL import Image
import os, shutil, json, yaml, zipfile, time
from collections import defaultdict

print(f'keras_hub : {keras_hub.__version__}')
print(f'keras     : {keras.__version__}')

# SAM3 sınıfı mevcut mu?
sam3_cls = [x for x in dir(keras_hub.models) if 'SAM3' in x]
print(f'SAM3 sınıfları: {sam3_cls}')

In [ ]:
# ── 2. Konfigürasyon ─────────────────────────────────────────────────────────

# ---------------------------------------------------------------------------
# HEDEF SINIFLAR — SAM3'e text prompt olarak gönderilir
# ---------------------------------------------------------------------------
TARGET_CLASSES = [
    'tank',
    'armored vehicle',
    'military truck',
    'artillery',
    'helicopter',
    'soldier',
]

CLASS_MAP  = {i: name for i, name in enumerate(TARGET_CLASSES)}
NAME_TO_ID = {name: i for i, name in CLASS_MAP.items()}

# ---------------------------------------------------------------------------
# SAM3 PARAMETRELERİ
# ---------------------------------------------------------------------------
SCORE_THRESHOLD = 0.5    # altındaki tespitler atlanır
MIN_AREA_RATIO  = 0.001  # görüntünün min %0.1'i
MAX_AREA_RATIO  = 0.90   # görüntünün max %90'ı

# ---------------------------------------------------------------------------
# GİRDİ / ÇIKTI
# ---------------------------------------------------------------------------
SAM3_PATH        = '/kaggle/input/models/keras/sam3/keras/sam3_pcs/1'
IMAGE_INPUT_DIR  = ''    # boş = otomatik bul
OUTPUT_DIR       = Path('/kaggle/working/sam3_labeled')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

print('Sınıflar:')
for i, c in CLASS_MAP.items():
    print(f'  [{i}] {c}')

In [ ]:
# ── 3. SAM3 Yükle ───────────────────────────────────────────────────────────
print(f'SAM3 yükleniyor: {SAM3_PATH}')
sam_model = keras_hub.models.SAM3PromptableConceptImageSegmenter.from_preset(SAM3_PATH)
print('SAM3 hazır ✅')

In [ ]:
# ── 4. Görüntü Dizinini Bul ──────────────────────────────────────────────────
def find_image_dir(base='/kaggle/input'):
    best_dir, best_count = Path(base), 0
    for dirpath, _, filenames in os.walk(base):
        imgs = [f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) > best_count:
            best_count, best_dir = len(imgs), Path(dirpath)
    return best_dir, best_count

if IMAGE_INPUT_DIR:
    img_dir = Path(IMAGE_INPUT_DIR)
else:
    img_dir, _ = find_image_dir()

all_images = sorted(p for p in img_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
print(f'📂 Dizin   : {img_dir}')
print(f'🖼️  Görüntü : {len(all_images):,}')

In [ ]:
# ── 5. Çıktı Klasörlerini Hazırla ────────────────────────────────────────────
img_out   = OUTPUT_DIR / 'images'
label_out = OUTPUT_DIR / 'labels'
img_out.mkdir(parents=True, exist_ok=True)
label_out.mkdir(parents=True, exist_ok=True)
print(f'Çıktı: {OUTPUT_DIR}')

In [ ]:
# ── 6. SAM3 Inference Fonksiyonu ─────────────────────────────────────────────
def run_sam3(image: Image.Image) -> list:
    """
    Tüm hedef sınıfları tek seferde sorgular.
    Her sınıf için SAM3'e text prompt + tam görüntü bbox gönderilir.
    Döner: [{class_id, class, bbox:[x1,y1,x2,y2], score}]
    """
    W, H = image.size
    img_array = np.array(image.convert('RGB')).astype('float32')
    n = len(TARGET_CLASSES)

    input_data = {
        'images'    : np.stack([img_array] * n),                    # (N, H, W, 3)
        'prompts'   : TARGET_CLASSES,                               # ["tank", ...]
        'boxes'     : np.tile([[[0, 0, W, H]]], (n, 1, 1)).astype('float32'),  # (N, 1, 4)
        'box_labels': np.ones((n, 1), dtype='float32'),
    }

    outputs   = sam_model.predict(input_data, verbose=0)
    scores    = outputs['scores']  # (N, num_queries)
    boxes_out = outputs['boxes']   # (N, num_queries, 4)  XYXY

    detections = []
    for i, class_name in enumerate(TARGET_CLASSES):
        best = int(np.argmax(scores[i]))
        score = float(scores[i][best])

        if score < SCORE_THRESHOLD:
            continue

        x1, y1, x2, y2 = boxes_out[i][best]
        x1, y1, x2, y2 = float(x1), float(y1), float(x2), float(y2)

        area_ratio = ((x2 - x1) * (y2 - y1)) / (W * H)
        if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
            continue

        detections.append({
            'class_id': NAME_TO_ID[class_name],
            'class'   : class_name,
            'bbox'    : [x1, y1, x2, y2],
            'score'   : score,
        })

    return detections

In [ ]:
# ── 7. YOLO Label Kaydet ─────────────────────────────────────────────────────
def save_yolo_labels(detections: list, img_w: int, img_h: int, path: Path) -> int:
    lines = []
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        cx = ((x1 + x2) / 2) / img_w
        cy = ((y1 + y2) / 2) / img_h
        bw = (x2 - x1)       / img_w
        bh = (y2 - y1)       / img_h
        cx = max(0.0, min(1.0, cx))
        cy = max(0.0, min(1.0, cy))
        bw = max(0.001, min(1.0, bw))
        bh = max(0.001, min(1.0, bh))
        lines.append(f"{d['class_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    if lines:
        path.write_text('\n'.join(lines))
    return len(lines)

In [ ]:
# ── 8. Ana İşleme Döngüsü ────────────────────────────────────────────────────
stats = {
    'total'      : len(all_images),
    'labeled'    : 0,
    'empty'      : 0,
    'errors'     : 0,
    'total_det'  : 0,
    'class_counts': defaultdict(int),
}

print(f'🚀 {len(all_images):,} görüntü işleniyor')
print(f'   Sınıflar  : {TARGET_CLASSES}')
print(f'   Score eşiği: {SCORE_THRESHOLD}')
print()

for i, img_path in enumerate(all_images):
    try:
        image = Image.open(img_path).convert('RGB')
        W, H  = image.size

        detections = run_sam3(image)

        dest_img   = img_out   / img_path.name
        label_path = label_out / (img_path.stem + '.txt')
        shutil.copy2(img_path, dest_img)
        n_det = save_yolo_labels(detections, W, H, label_path)

        if n_det > 0:
            stats['labeled']   += 1
            stats['total_det'] += n_det
            for d in detections:
                stats['class_counts'][d['class']] += 1
        else:
            stats['empty'] += 1

    except Exception as e:
        stats['errors'] += 1
        print(f'  ❌ [{img_path.name}]: {e}')
        continue

    if (i + 1) % 10 == 0 or i == len(all_images) - 1:
        pct = (i + 1) / len(all_images) * 100
        print(f'  [{i+1:>5}/{len(all_images)}] {pct:5.1f}%  '
              f'labeled={stats["labeled"]}  empty={stats["empty"]}  errors={stats["errors"]}')

print('\n✅ Tamamlandı!')

In [ ]:
# ── 9. İstatistikler ─────────────────────────────────────────────────────────
total   = stats['total']
labeled = stats['labeled']

print('=' * 50)
print('📊 SONUÇLAR')
print('=' * 50)
print(f'Toplam görüntü    : {total:,}')
print(f'Etiketlenen       : {labeled:,}  ({labeled/max(1,total)*100:.1f}%)')
print(f'Boş               : {stats["empty"]:,}')
print(f'Hata              : {stats["errors"]:,}')
print(f'Toplam tespit     : {stats["total_det"]:,}')
print()
print('─── Sınıf Dağılımı ───')
if stats['class_counts']:
    max_c = max(stats['class_counts'].values())
    for cls, cnt in sorted(stats['class_counts'].items(), key=lambda x: -x[1]):
        bar = '█' * int(cnt / max_c * 35)
        print(f'  {cls:<25} {cnt:>5,}  {bar}')
print('=' * 50)

In [ ]:
# ── 10. Görselleştirme ───────────────────────────────────────────────────────
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PALETTE = [
    '#00e676','#40c4ff','#ff6d00','#ea80fc',
    '#ffd740','#69f0ae','#ff4081','#80d8ff',
]

def draw_sample(ax, img_path, label_path):
    img = Image.open(img_path)
    ax.imshow(img); ax.axis('off')
    ax.set_title(img_path.name, fontsize=7, pad=2)
    if not label_path.exists() or label_path.stat().st_size == 0:
        return
    W, H = img.size
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        cid   = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:5])
        x1, y1 = (cx - bw/2)*W, (cy - bh/2)*H
        color = PALETTE[cid % len(PALETTE)]
        ax.add_patch(patches.Rectangle((x1,y1), bw*W, bh*H,
                     lw=2, edgecolor=color, facecolor='none'))
        ax.text(x1+2, y1-4, CLASS_MAP.get(cid, f'cls{cid}'),
                color=color, fontsize=7,
                bbox=dict(boxstyle='square,pad=0.1', fc='black', alpha=0.75, lw=0))

labeled_imgs = [
    p for p in img_out.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
    and (label_out / (p.stem+'.txt')).exists()
    and (label_out / (p.stem+'.txt')).stat().st_size > 0
]
n_show = min(12, len(labeled_imgs))
sample = random.sample(labeled_imgs, n_show) if n_show else []

if sample:
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5))
    for i, ax in enumerate(np.array(axes).flatten()):
        if i < len(sample):
            draw_sample(ax, sample[i], label_out/(sample[i].stem+'.txt'))
        else:
            ax.axis('off')
    plt.suptitle('SAM3 Auto-Label Örnekleri', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'preview.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print('Etiketlenmiş görüntü yok.')

In [ ]:
# ── 11. data.yaml ────────────────────────────────────────────────────────────
yaml_content = {
    'path' : str(OUTPUT_DIR),
    'train': 'images',
    'val'  : 'images',
    'nc'   : len(TARGET_CLASSES),
    'names': TARGET_CLASSES,
}
yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)
print(yaml_path.read_text())

In [ ]:
# ── 12. ZIP Paketle ──────────────────────────────────────────────────────────
zip_path = Path('/kaggle/working') / f'sam3_labeled_{int(time.time())}.zip'
files    = [f for f in OUTPUT_DIR.rglob('*') if f.is_file()]

print(f'📦 {len(files):,} dosya paketleniyor...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in files:
        zf.write(f, f.relative_to(OUTPUT_DIR))

print(f'✅ {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
print('→ Kaggle Output sekmesinden indir')
print('→ unilabellm\'e yükle → gözden geçir → export → retrain')

## İpuçları

| Parametre | Varsayılan | Az tespit | Çok tespit |
|-----------|-----------|-----------|------------|
| `SCORE_THRESHOLD` | 0.50 | ↑ 0.70 | ↓ 0.30 |
| `MIN_AREA_RATIO`  | 0.001 | ↑ 0.01 | ↓ 0.0005 |

**Sınıfları düzenle:** `TARGET_CLASSES` listesini kendi veri setine göre değiştir.  
SAM3 Türkçe prompt da anlıyor — `'tank'` yerine `'zırhlı araç'` yazabilirsin.